# Análise Exploratória de Dados (EDA)

## Objetivo

Investigar a deterioração recente observada no desempenho comercial do e-commerce entre maio e junho de 2018.

## Fenômeno investigado

Entre maio e junho de 2018, a receita de produtos apresentou redução de 13,19%, acompanhada por redução de 10,11% no número de pedidos e de 3,42% no ticket médio.

Esse fenômeno foi identificado durante a etapa de Data Profiling e será utilizado como foco da investigação exploratória.

## Pergunta central da EDA

Onde e como a deterioração observada entre maio e junho de 2018 se manifesta nos dados?

## Abordagem

A EDA buscará:

- localizar a concentração da deterioração;
- decompor seus principais componentes;
- identificar padrões relevantes;
- investigar as dimensões que apresentarem evidências de participação no fenômeno;
- formular hipóteses para posterior análise estatística.

Os resultados desta etapa serão tratados como evidências exploratórias e não serão interpretados como evidência de causalidade.

In [1]:
import pandas as pd

In [2]:
orders_path = "../data/raw/olist_orders_dataset.csv"
order_items_path = "../data/raw/olist_order_items_dataset.csv"

In [3]:
orders = pd.read_csv(orders_path)

In [4]:
order_items = pd.read_csv(order_items_path)

In [5]:
print("ORDERS:", orders.shape)
print("COLUNAS:", orders.columns.tolist())

print("\nORDER_ITEMS:", order_items.shape)
print("COLUNAS:", order_items.columns.tolist())

ORDERS: (99441, 8)
COLUNAS: ['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date', 'order_delivered_customer_date', 'order_estimated_delivery_date']

ORDER_ITEMS: (112650, 7)
COLUNAS: ['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']


In [6]:
print("Orders — order_id únicos:", orders["order_id"].nunique())
print("Orders — linhas:", len(orders))

print("\nOrder Items — order_id únicos:", order_items["order_id"].nunique())
print("Order Items — linhas:", len(order_items))

Orders — order_id únicos: 99441
Orders — linhas: 99441

Order Items — order_id únicos: 98666
Order Items — linhas: 112650


In [7]:
order_items["order_id"].isin(orders["order_id"]).value_counts()

order_id
True    112650
Name: count, dtype: int64

In [8]:
orders["order_purchase_timestamp"].dtype

<StringDtype(storage='python', na_value=nan)>

In [9]:
orders["order_purchase_timestamp"] = pd.to_datetime(
    orders["order_purchase_timestamp"]
)

In [10]:
orders["order_purchase_timestamp"].dtype

dtype('<M8[us]')

In [11]:
orders["order_purchase_timestamp"].isna().sum()

np.int64(0)

In [12]:
print("Tipo de price:", order_items["price"].dtype)
print("Valores nulos:", order_items["price"].isna().sum())

Tipo de price: float64
Valores nulos: 0


In [13]:
order_items.duplicated(
    subset=["order_id", "order_item_id"]
).sum()

np.int64(0)

In [14]:
eda_base = order_items.merge(
    orders[["order_id", "order_purchase_timestamp", "order_status"]],
    on="order_id",
    how="left"
)

In [15]:
eda_base.shape

(112650, 9)

In [16]:
eda_base[
    ["order_purchase_timestamp", "order_status"]
].isna().sum()

order_purchase_timestamp    0
order_status                0
dtype: int64

In [17]:
print("Data inicial:", eda_base["order_purchase_timestamp"].min())
print("Data final:", eda_base["order_purchase_timestamp"].max())

Data inicial: 2016-09-04 21:15:19
Data final: 2018-09-03 09:06:57


In [18]:
eda_base["purchase_month"] = (
    eda_base["order_purchase_timestamp"]
    .dt.to_period("M")
)

In [19]:
eda_base["purchase_month"].head()

0    2017-09
1    2017-04
2    2018-01
3    2018-08
4    2017-02
Name: purchase_month, dtype: period[M]

In [20]:
receita_mensal = (
    eda_base
    .groupby("purchase_month")["price"]
    .sum()
)

receita_mensal

purchase_month
2016-09        267.36
2016-10      49507.66
2016-12         10.90
2017-01     120312.87
2017-02     247303.02
2017-03     374344.30
2017-04     359927.23
2017-05     506071.14
2017-06     433038.60
2017-07     498031.48
2017-08     573971.68
2017-09     624401.69
2017-10     664219.43
2017-11    1010271.37
2017-12     743914.17
2018-01     950030.36
2018-02     844178.71
2018-03     983213.44
2018-04     996647.75
2018-05     996517.68
2018-06     865124.31
2018-07     895507.22
2018-08     854686.33
2018-09        145.00
Freq: M, Name: price, dtype: float64

In [21]:
receita_maio = receita_mensal.loc["2018-05"]
receita_junho = receita_mensal.loc["2018-06"]

variacao_receita = (
    (receita_junho - receita_maio)
    / receita_maio
) * 100

variacao_receita

np.float64(-13.185252267676775)

In [22]:
pedidos_mensais = (
    eda_base
    .groupby("purchase_month")["order_id"]
    .nunique()
)

pedidos_mensais

purchase_month
2016-09       3
2016-10     308
2016-12       1
2017-01     789
2017-02    1733
2017-03    2641
2017-04    2391
2017-05    3660
2017-06    3217
2017-07    3969
2017-08    4293
2017-09    4243
2017-10    4568
2017-11    7451
2017-12    5624
2018-01    7220
2018-02    6694
2018-03    7188
2018-04    6934
2018-05    6853
2018-06    6160
2018-07    6273
2018-08    6452
2018-09       1
Freq: M, Name: order_id, dtype: int64

In [23]:
ticket_mensal = receita_mensal / pedidos_mensais

ticket_mensal

purchase_month
2016-09     89.120000
2016-10    160.739156
2016-12     10.900000
2017-01    152.487795
2017-02    142.702262
2017-03    141.743393
2017-04    150.534182
2017-05    138.270803
2017-06    134.609450
2017-07    125.480343
2017-08    133.699436
2017-09    147.160427
2017-10    145.407056
2017-11    135.588695
2017-12    132.274924
2018-01    131.583152
2018-02    126.109756
2018-03    136.785398
2018-04    143.733451
2018-05    145.413349
2018-06    140.442258
2018-07    142.755814
2018-08    132.468433
2018-09    145.000000
Freq: M, dtype: float64

In [24]:
receita_reconstruida = pedidos_mensais * ticket_mensal

pd.DataFrame({
    "receita_original": receita_mensal,
    "receita_reconstruida": receita_reconstruida,
    "diferenca": receita_mensal - receita_reconstruida
}).loc[["2018-05", "2018-06"]]

,receita_original,receita_reconstruida,diferenca
purchase_month,,,
2018-05,996517.68,996517.68,0.0
2018-06,865124.31,865124.31,0.0


In [25]:
products_path = "../data/raw/olist_products_dataset.csv"

products = pd.read_csv(products_path)

products.shape

(32951, 9)

In [26]:
products.columns.tolist()

['product_id',
 'product_category_name',
 'product_name_lenght',
 'product_description_lenght',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

In [27]:
eda_base = eda_base.merge(
    products[["product_id", "product_category_name"]],
    on="product_id",
    how="left"
)

eda_base.shape

(112650, 11)

In [28]:
eda_base.columns.tolist()

['order_id',
 'order_item_id',
 'product_id',
 'seller_id',
 'shipping_limit_date',
 'price',
 'freight_value',
 'order_purchase_timestamp',
 'order_status',
 'purchase_month',
 'product_category_name']

In [29]:
eda_base["product_category_name"].isna().sum()

np.int64(1603)

In [30]:
receita_categoria = (
    eda_base[
        eda_base["purchase_month"].isin(["2018-05", "2018-06"])
    ]
    .groupby(
        ["purchase_month", "product_category_name"],
        dropna=False
    )["price"]
    .sum()
    .unstack(fill_value=0)
)

receita_categoria

product_category_name
purchase_month


In [31]:
eda_base["purchase_month"].dtype

period[M]

In [32]:
meses_analise = pd.PeriodIndex(
    ["2018-05", "2018-06"],
    freq="M"
)

meses_analise

PeriodIndex(['2018-05', '2018-06'], dtype='period[M]')

In [33]:
receita_categoria = (
    eda_base[
        eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby(
        ["purchase_month", "product_category_name"],
        dropna=False
    )["price"]
    .sum()
    .unstack(fill_value=0)
)

receita_categoria

product_category_name,agro_industria_e_comercio,alimentos,alimentos_bebidas,artes,artes_e_artesanato,artigos_de_festas,artigos_de_natal,audio,automotivo,bebes,...,pet_shop,portateis_casa_forno_e_cafe,portateis_cozinha_e_preparadores_de_alimentos,relogios_presentes,sinalizacao_e_seguranca,tablets_impressao_imagem,telefonia,telefonia_fixa,utilidades_domesticas,NaN
purchase_month,,,,,,,,,,,,,,,,,,,,,
2018-05,3279.75,1542.93,1239.44,2691.86,89.8,100.7,524.53,2697.68,40928.14,34003.60,...,11708.46,10161.6,1958.0,123872.66,683.50,49.9,21608.26,1946.10,63003.57,3136.94
2018-06,3515.80,1371.18,293.85,2500.14,0.0,84.8,306.19,1830.69,45407.49,30755.12,...,19491.78,3499.0,0.0,86885.56,5863.18,50.0,22776.82,2080.32,56825.45,2387.13


In [34]:
variacao_categoria = (
    receita_categoria.loc["2018-06"]
    - receita_categoria.loc["2018-05"]
)

variacao_categoria.sort_values()

product_category_name
relogios_presentes        -36987.10
ferramentas_jardim        -19844.47
esporte_lazer             -13813.10
moveis_decoracao          -12070.76
cool_stuff                -10722.74
                             ...   
automotivo                  4479.35
sinalizacao_e_seguranca     5179.68
eletroportateis             5401.71
pet_shop                    7783.32
beleza_saude               11448.46
Length: 67, dtype: float64

In [35]:
variacao_percentual_categoria = (
    (
        receita_categoria.loc["2018-06"]
        - receita_categoria.loc["2018-05"]
    )
    / receita_categoria.loc["2018-05"]
) * 100

variacao_percentual_categoria.sort_values()

product_category_name
artes_e_artesanato                              -100.000000
portateis_cozinha_e_preparadores_de_alimentos   -100.000000
musica                                           -85.037037
cine_foto                                        -80.861023
alimentos_bebidas                                -76.291712
                                                    ...    
fashion_roupa_masculina                          244.488038
flores                                           558.536585
sinalizacao_e_seguranca                          757.817118
casa_conforto_2                                         inf
fashion_roupa_infanto_juvenil                           inf
Length: 67, dtype: float64

In [36]:
analise_categoria = pd.DataFrame({
    "receita_maio": receita_categoria.loc["2018-05"],
    "receita_junho": receita_categoria.loc["2018-06"],
    "variacao_absoluta": variacao_categoria,
    "variacao_percentual": variacao_percentual_categoria
})

analise_categoria.sort_values(
    "variacao_absoluta"
)

,receita_maio,receita_junho,variacao_absoluta,variacao_percentual
product_category_name,,,,
relogios_presentes,123872.66,86885.56,-36987.10,-29.858970
ferramentas_jardim,37863.18,18018.71,-19844.47,-52.410997
esporte_lazer,59664.11,45851.01,-13813.10,-23.151439
moveis_decoracao,54260.98,42190.22,-12070.76,-22.245746
cool_stuff,30039.11,19316.37,-10722.74,-35.695931
...,...,...,...,...
automotivo,40928.14,45407.49,4479.35,10.944426
sinalizacao_e_seguranca,683.50,5863.18,5179.68,757.817118
eletroportateis,10620.99,16022.70,5401.71,50.858818


In [37]:
pedidos_categoria = (
    eda_base[
        eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby(
        ["purchase_month", "product_category_name"],
        dropna=False
    )["order_id"]
    .nunique()
    .unstack(fill_value=0)
)

pedidos_categoria

product_category_name,agro_industria_e_comercio,alimentos,alimentos_bebidas,artes,artes_e_artesanato,artigos_de_festas,artigos_de_natal,audio,automotivo,bebes,...,pet_shop,portateis_casa_forno_e_cafe,portateis_cozinha_e_preparadores_de_alimentos,relogios_presentes,sinalizacao_e_seguranca,tablets_impressao_imagem,telefonia,telefonia_fixa,utilidades_domesticas,NaN
purchase_month,,,,,,,,,,,,,,,,,,,,,
2018-05,8,36,15,35,1,3,6,20,298,224,...,127,9,2,596,6,1,243,12,493,38
2018-06,10,24,9,28,0,2,6,19,272,207,...,151,3,0,448,18,1,205,15,470,22


In [38]:
ticket_categoria = (
    receita_categoria
    / pedidos_categoria
)

ticket_categoria

product_category_name,agro_industria_e_comercio,alimentos,alimentos_bebidas,artes,artes_e_artesanato,artigos_de_festas,artigos_de_natal,audio,automotivo,bebes,...,pet_shop,portateis_casa_forno_e_cafe,portateis_cozinha_e_preparadores_de_alimentos,relogios_presentes,sinalizacao_e_seguranca,tablets_impressao_imagem,telefonia,telefonia_fixa,utilidades_domesticas,NaN
purchase_month,,,,,,,,,,,,,,,,,,,,,
2018-05,409.96875,42.859167,82.629333,76.910286,89.8,33.566667,87.421667,134.884000,137.342752,151.801786,...,92.192598,1129.066667,979.0,207.840034,113.916667,49.9,88.922881,162.175,127.796288,82.551053
2018-06,351.58000,57.132500,32.650000,89.290714,NaN,42.400000,51.031667,96.352105,166.939301,148.575459,...,129.084636,1166.333333,NaN,193.940982,325.732222,50.0,111.106439,138.688,120.905213,108.505909


In [39]:
analise_categoria = analise_categoria.copy()

analise_categoria["pedidos_maio"] = pedidos_categoria.loc["2018-05"]
analise_categoria["pedidos_junho"] = pedidos_categoria.loc["2018-06"]

analise_categoria["ticket_maio"] = ticket_categoria.loc["2018-05"]
analise_categoria["ticket_junho"] = ticket_categoria.loc["2018-06"]

analise_categoria

,receita_maio,receita_junho,variacao_absoluta,variacao_percentual,pedidos_maio,pedidos_junho,ticket_maio,ticket_junho
product_category_name,,,,,,,,
agro_industria_e_comercio,3279.75,3515.80,236.05,7.197195,8,10,409.968750,351.580000
alimentos,1542.93,1371.18,-171.75,-11.131419,36,24,42.859167,57.132500
alimentos_bebidas,1239.44,293.85,-945.59,-76.291712,15,9,82.629333,32.650000
artes,2691.86,2500.14,-191.72,-7.122213,35,28,76.910286,89.290714
artes_e_artesanato,89.80,0.00,-89.80,-100.000000,1,0,89.800000,NaN
...,...,...,...,...,...,...,...,...
tablets_impressao_imagem,49.90,50.00,0.10,0.200401,1,1,49.900000,50.000000
telefonia,21608.26,22776.82,1168.56,5.407932,243,205,88.922881,111.106439
telefonia_fixa,1946.10,2080.32,134.22,6.896871,12,15,162.175000,138.688000


In [40]:
categorias_com_queda = (
    analise_categoria[
        analise_categoria["variacao_absoluta"] < 0
    ]
    .sort_values("variacao_absoluta")
)

categorias_com_queda[
    [
        "receita_maio",
        "receita_junho",
        "variacao_absoluta",
        "variacao_percentual",
        "pedidos_maio",
        "pedidos_junho",
        "ticket_maio",
        "ticket_junho"
    ]
]

,receita_maio,receita_junho,variacao_absoluta,variacao_percentual,pedidos_maio,pedidos_junho,ticket_maio,ticket_junho
product_category_name,,,,,,,,
relogios_presentes,123872.66,86885.56,-36987.10,-29.858970,596,448,207.840034,193.940982
ferramentas_jardim,37863.18,18018.71,-19844.47,-52.410997,230,149,164.622522,120.930940
esporte_lazer,59664.11,45851.01,-13813.10,-23.151439,452,377,132.000243,121.620716
moveis_decoracao,54260.98,42190.22,-12070.76,-22.245746,462,318,117.448009,132.673648
cool_stuff,30039.11,19316.37,-10722.74,-35.695931,169,100,177.746213,193.163700
brinquedos,31767.30,21221.45,-10545.85,-33.197187,214,177,148.445327,119.895198
informatica_acessorios,51515.73,42326.45,-9189.28,-17.837814,409,358,125.955330,118.230307
moveis_escritorio,21005.86,12725.58,-8280.28,-39.418905,83,60,253.082651,212.093000
eletronicos,16989.29,9664.57,-7324.72,-43.113750,223,174,76.185157,55.543506


In [41]:
id="c8m421"
perdas_categoria = categorias_com_queda.copy()

perdas_categoria["perda_positiva"] = (
    perdas_categoria["variacao_absoluta"] * -1
)

perdas_categoria["participacao_perdas"] = (
    perdas_categoria["perda_positiva"]
    / perdas_categoria["perda_positiva"].sum()
) * 100

perdas_categoria["perda_acumulada"] = (
    perdas_categoria["perda_positiva"].cumsum()
)

perdas_categoria["participacao_acumulada"] = (
    perdas_categoria["perda_acumulada"]
    / perdas_categoria["perda_positiva"].sum()
) * 100

In [42]:
id="r73164"
perdas_categoria[
    [
        "variacao_absoluta",
        "perda_positiva",
        "participacao_perdas",
        "perda_acumulada",
        "participacao_acumulada"
    ]
].head(10)

,variacao_absoluta,perda_positiva,participacao_perdas,perda_acumulada,participacao_acumulada
product_category_name,,,,,
relogios_presentes,-36987.10,36987.10,20.366980,36987.10,20.366980
ferramentas_jardim,-19844.47,19844.47,10.927375,56831.57,31.294356
esporte_lazer,-13813.10,13813.10,7.606196,70644.67,38.900552
moveis_decoracao,-12070.76,12070.76,6.646775,82715.43,45.547327
cool_stuff,-10722.74,10722.74,5.904487,93438.17,51.451813
brinquedos,-10545.85,10545.85,5.807082,103984.02,57.258895
informatica_acessorios,-9189.28,9189.28,5.060085,113173.30,62.318980
moveis_escritorio,-8280.28,8280.28,4.559544,121453.58,66.878524
eletronicos,-7324.72,7324.72,4.033364,128778.30,70.911888


In [43]:
categorias_prioritarias = perdas_categoria.head(10).copy()

categorias_prioritarias[
    [
        "receita_maio",
        "receita_junho",
        "variacao_absoluta",
        "variacao_percentual",
        "pedidos_maio",
        "pedidos_junho",
        "ticket_maio",
        "ticket_junho"
    ]
]

,receita_maio,receita_junho,variacao_absoluta,variacao_percentual,pedidos_maio,pedidos_junho,ticket_maio,ticket_junho
product_category_name,,,,,,,,
relogios_presentes,123872.66,86885.56,-36987.10,-29.858970,596,448,207.840034,193.940982
ferramentas_jardim,37863.18,18018.71,-19844.47,-52.410997,230,149,164.622522,120.930940
esporte_lazer,59664.11,45851.01,-13813.10,-23.151439,452,377,132.000243,121.620716
moveis_decoracao,54260.98,42190.22,-12070.76,-22.245746,462,318,117.448009,132.673648
cool_stuff,30039.11,19316.37,-10722.74,-35.695931,169,100,177.746213,193.163700
brinquedos,31767.30,21221.45,-10545.85,-33.197187,214,177,148.445327,119.895198
informatica_acessorios,51515.73,42326.45,-9189.28,-17.837814,409,358,125.955330,118.230307
moveis_escritorio,21005.86,12725.58,-8280.28,-39.418905,83,60,253.082651,212.093000
eletronicos,16989.29,9664.57,-7324.72,-43.113750,223,174,76.185157,55.543506


In [44]:
analise_mecanismos = categorias_prioritarias.copy()

analise_mecanismos["variacao_pedidos_percentual"] = (
    (
        analise_mecanismos["pedidos_junho"]
        - analise_mecanismos["pedidos_maio"]
    )
    / analise_mecanismos["pedidos_maio"]
) * 100

analise_mecanismos["variacao_ticket_percentual"] = (
    (
        analise_mecanismos["ticket_junho"]
        - analise_mecanismos["ticket_maio"]
    )
    / analise_mecanismos["ticket_maio"]
) * 100

analise_mecanismos[
    [
        "variacao_absoluta",
        "variacao_percentual",
        "variacao_pedidos_percentual",
        "variacao_ticket_percentual"
    ]
]

,variacao_absoluta,variacao_percentual,variacao_pedidos_percentual,variacao_ticket_percentual
product_category_name,,,,
relogios_presentes,-36987.10,-29.858970,-24.832215,-6.687379
ferramentas_jardim,-19844.47,-52.410997,-35.217391,-26.540465
esporte_lazer,-13813.10,-23.151439,-16.592920,-7.863264
moveis_decoracao,-12070.76,-22.245746,-31.168831,12.963727
cool_stuff,-10722.74,-35.695931,-40.828402,8.673876
brinquedos,-10545.85,-33.197187,-17.289720,-19.232757
informatica_acessorios,-9189.28,-17.837814,-12.469438,-6.133145
moveis_escritorio,-8280.28,-39.418905,-27.710843,-16.196152
eletronicos,-7324.72,-43.113750,-21.973094,-27.094059


In [45]:
categorias_foco = categorias_prioritarias.index.tolist()

categorias_foco

['relogios_presentes',
 'ferramentas_jardim',
 'esporte_lazer',
 'moveis_decoracao',
 'cool_stuff',
 'brinquedos',
 'informatica_acessorios',
 'moveis_escritorio',
 'eletronicos',
 'portateis_casa_forno_e_cafe']

In [47]:
receita_produto = (
    eda_base[
        eda_base["product_category_name"].isin(categorias_foco)
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby(
        [
            "purchase_month",
            "product_category_name",
            "product_id"
        ]
    )["price"]
    .sum()
    .unstack(fill_value=0)
)

In [48]:
receita_produto.shape

(20, 2413)

In [49]:
receita_produto = (
    eda_base[
        eda_base["product_category_name"].isin(categorias_foco)
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby(
        [
            "product_category_name",
            "product_id",
            "purchase_month"
        ]
    )["price"]
    .sum()
    .unstack("purchase_month", fill_value=0)
    .reset_index()
)

In [50]:
receita_produto.shape

(2413, 4)

In [53]:
receita_produto["variacao_absoluta"] = (
    receita_produto[meses_analise[1]]
    - receita_produto[meses_analise[0]]
)

receita_produto["variacao_percentual"] = (
    receita_produto["variacao_absoluta"]
    / receita_produto[meses_analise[0]]
) * 100

In [54]:
produtos_com_queda = (
    receita_produto[
        receita_produto["variacao_absoluta"] < 0
    ]
    .sort_values("variacao_absoluta")
)

In [55]:
produtos_com_queda[
    [
        "product_category_name",
        "product_id",
        meses_analise[0],
        meses_analise[1],
        "variacao_absoluta",
        "variacao_percentual"
    ]
].head(20)

purchase_month,product_category_name,product_id,2018-05,2018-06,variacao_absoluta,variacao_percentual
2174,relogios_presentes,53b36df67ebb7c41585e8d54d6772e08,11820.20,1071.0,-10749.20,-90.939240
1819,moveis_decoracao,aca2eb7d00ea1a7b8ebd4e68314663af,6430.80,179.8,-6251.00,-97.204080
1420,informatica_acessorios,af751fe756699bddfce4fdd67242fc2a,5040.00,0.0,-5040.00,-100.000000
2347,relogios_presentes,d285360f29ac7fd97640bf0baef03de0,5956.30,983.6,-4972.70,-83.486393
377,eletronicos,0abc19e41fdc9d6894b8cae651619253,4941.00,0.0,-4941.00,-100.000000
2182,relogios_presentes,57c9362442482cd2451f2eabf468e532,4955.97,869.0,-4086.97,-82.465592
1381,informatica_acessorios,89b121bee266dcd25688a1ba72eefb61,3995.00,0.0,-3995.00,-100.000000
1056,ferramentas_jardim,0563d4cc419141eab2e5b438046e844e,3930.00,0.0,-3930.00,-100.000000
2213,relogios_presentes,6f3b5b605d91b7439c5e3f5a8dffeea7,4355.00,457.9,-3897.10,-89.485649
1292,informatica_acessorios,3dd2a17168ec895c781a9191c1e95ad7,7794.80,4047.3,-3747.50,-48.076923


In [56]:
receita_produto.columns

Index(['product_category_name',            'product_id',
                       2018-05,                 2018-06,
           'variacao_absoluta',   'variacao_percentual'],
      dtype='object', name='purchase_month')

In [57]:
produtos_com_queda["perda_receita"] = (
    produtos_com_queda["variacao_absoluta"] * -1
)

In [58]:
produtos_com_queda["participacao_perdas"] = (
    produtos_com_queda["perda_receita"]
    / produtos_com_queda["perda_receita"].sum()
) * 100

produtos_com_queda["perda_acumulada"] = (
    produtos_com_queda["perda_receita"].cumsum()
)

produtos_com_queda["participacao_acumulada"] = (
    produtos_com_queda["perda_acumulada"]
    / produtos_com_queda["perda_receita"].sum()
) * 100

In [59]:
produtos_com_queda[
    produtos_com_queda["participacao_acumulada"] <= 50
][
    [
        "product_category_name",
        "product_id",
        "perda_receita",
        "participacao_acumulada"
    ]
].tail(1)

purchase_month,product_category_name,product_id,perda_receita,participacao_acumulada
2082,relogios_presentes,101a896b1137f70a7bbb2b9fac61458d,640.0,49.938814


In [60]:
quantidade_produtos_50 = (
    produtos_com_queda[
        produtos_com_queda["participacao_acumulada"] <= 50
    ]
    .shape[0]
)

quantidade_produtos_50

104

In [61]:
quantidade_produtos_75 = (
    produtos_com_queda[
        produtos_com_queda["participacao_acumulada"] <= 75
    ]
    .shape[0]
)

quantidade_produtos_75

325

In [62]:
quantidade_produtos_90 = (
    produtos_com_queda[
        produtos_com_queda["participacao_acumulada"] <= 90
    ]
    .shape[0]
)

quantidade_produtos_90

634

In [63]:
produtos_top_50 = (
    produtos_com_queda[
        produtos_com_queda["participacao_acumulada"] <= 50
    ]
)

produtos_top_50["product_category_name"].value_counts()

product_category_name
relogios_presentes             33
ferramentas_jardim             13
cool_stuff                     13
esporte_lazer                  11
informatica_acessorios         10
brinquedos                      9
moveis_escritorio               6
moveis_decoracao                4
eletronicos                     3
portateis_casa_forno_e_cafe     2
Name: count, dtype: int64

In [64]:
perda_top_50_categoria = (
    produtos_top_50
    .groupby("product_category_name")["perda_receita"]
    .sum()
    .sort_values(ascending=False)
)

perda_top_50_categoria

product_category_name
relogios_presentes             62444.92
informatica_acessorios         21860.26
ferramentas_jardim             20634.52
cool_stuff                     13592.32
esporte_lazer                  11070.80
moveis_decoracao                8841.30
moveis_escritorio               8324.38
brinquedos                      7543.67
eletronicos                     7130.88
portateis_casa_forno_e_cafe     6833.60
Name: perda_receita, dtype: float64

In [65]:
pedidos_produto = (
    eda_base[
        eda_base["product_category_name"].isin(categorias_foco)
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby(
        [
            "product_category_name",
            "product_id",
            "purchase_month"
        ]
    )["order_id"]
    .nunique()
    .unstack("purchase_month", fill_value=0)
    .reset_index()
)

In [66]:
pedidos_produto.shape

(2413, 4)

In [67]:
pedidos_produto["variacao_pedidos"] = (
    pedidos_produto[meses_analise[1]]
    - pedidos_produto[meses_analise[0]]
)

pedidos_produto["variacao_pedidos_percentual"] = (
    pedidos_produto["variacao_pedidos"]
    / pedidos_produto[meses_analise[0]]
) * 100

In [68]:
ticket_produto = (
    receita_produto[meses_analise]
    / pedidos_produto[meses_analise]
)

In [69]:
ticket_produto.shape

(2413, 2)

In [71]:
analise_produto = (
    receita_produto[
        [
            "product_category_name",
            "product_id",
            meses_analise[0],
            meses_analise[1],
            "variacao_absoluta",
            "variacao_percentual"
        ]
    ]
    .merge(
        pedidos_produto[
            [
                "product_category_name",
                "product_id",
                meses_analise[0],
                meses_analise[1],
                "variacao_pedidos",
                "variacao_pedidos_percentual"
            ]
        ],
        on=["product_category_name", "product_id"],
        suffixes=("_receita", "_pedidos")
    )
)

In [72]:
analise_produto.shape

(2413, 10)

In [73]:
analise_produto["ticket_maio"] = ticket_produto[meses_analise[0]].values
analise_produto["ticket_junho"] = ticket_produto[meses_analise[1]].values

In [74]:
analise_produto["variacao_ticket"] = (
    analise_produto["ticket_junho"]
    - analise_produto["ticket_maio"]
)

analise_produto["variacao_ticket_percentual"] = (
    analise_produto["variacao_ticket"]
    / analise_produto["ticket_maio"]
) * 100

In [75]:
analise_produto_top_50 = (
    analise_produto[
        analise_produto["product_id"].isin(
            produtos_top_50["product_id"]
        )
    ]
    .copy()
)

In [76]:
analise_produto_top_50[
    [
        "product_category_name",
        "product_id",
        "2018-05_receita",
        "2018-06_receita",
        "variacao_absoluta",
        "variacao_percentual",
        "2018-05_pedidos",
        "2018-06_pedidos",
        "variacao_pedidos",
        "variacao_pedidos_percentual",
        "ticket_maio",
        "ticket_junho",
        "variacao_ticket",
        "variacao_ticket_percentual"
    ]
].sort_values(
    "variacao_absoluta"
).head(20)

purchase_month,product_category_name,product_id,2018-05_receita,2018-06_receita,variacao_absoluta,variacao_percentual,2018-05_pedidos,2018-06_pedidos,variacao_pedidos,variacao_pedidos_percentual,ticket_maio,ticket_junho,variacao_ticket,variacao_ticket_percentual
2174,relogios_presentes,53b36df67ebb7c41585e8d54d6772e08,11820.20,1071.0,-10749.20,-90.939240,107,9,-98,-91.588785,110.469159,119.000000,8.530841,7.722374
1819,moveis_decoracao,aca2eb7d00ea1a7b8ebd4e68314663af,6430.80,179.8,-6251.00,-97.204080,74,1,-73,-98.648649,86.902703,179.800000,92.897297,106.898053
1420,informatica_acessorios,af751fe756699bddfce4fdd67242fc2a,5040.00,0.0,-5040.00,-100.000000,3,0,-3,-100.000000,1680.000000,NaN,NaN,NaN
2347,relogios_presentes,d285360f29ac7fd97640bf0baef03de0,5956.30,983.6,-4972.70,-83.486393,23,4,-19,-82.608696,258.969565,245.900000,-13.069565,-5.046757
377,eletronicos,0abc19e41fdc9d6894b8cae651619253,4941.00,0.0,-4941.00,-100.000000,2,0,-2,-100.000000,2470.500000,NaN,NaN,NaN
2182,relogios_presentes,57c9362442482cd2451f2eabf468e532,4955.97,869.0,-4086.97,-82.465592,5,1,-4,-80.000000,991.194000,869.000000,-122.194000,-12.327960
1381,informatica_acessorios,89b121bee266dcd25688a1ba72eefb61,3995.00,0.0,-3995.00,-100.000000,46,0,-46,-100.000000,86.847826,NaN,NaN,NaN
1056,ferramentas_jardim,0563d4cc419141eab2e5b438046e844e,3930.00,0.0,-3930.00,-100.000000,1,0,-1,-100.000000,3930.000000,NaN,NaN,NaN
2213,relogios_presentes,6f3b5b605d91b7439c5e3f5a8dffeea7,4355.00,457.9,-3897.10,-89.485649,30,3,-27,-90.000000,145.166667,152.633333,7.466667,5.143513
1292,informatica_acessorios,3dd2a17168ec895c781a9191c1e95ad7,7794.80,4047.3,-3747.50,-48.076923,44,27,-17,-38.636364,177.154545,149.900000,-27.254545,-15.384615


In [77]:
analise_produto_top_50["mecanismo_queda"] = "não classificado"

analise_produto_top_50.loc[
    (
        (analise_produto_top_50["variacao_pedidos"] < 0)
        & (analise_produto_top_50["variacao_ticket"] < 0)
    ),
    "mecanismo_queda"
] = "pedidos ↓ + ticket ↓"

analise_produto_top_50.loc[
    (
        (analise_produto_top_50["variacao_pedidos"] < 0)
        & (analise_produto_top_50["variacao_ticket"] >= 0)
    ),
    "mecanismo_queda"
] = "pedidos ↓ + ticket ↑/estável"

analise_produto_top_50.loc[
    (
        (analise_produto_top_50["variacao_pedidos"] < 0)
        & (analise_produto_top_50["ticket_junho"].isna())
    ),
    "mecanismo_queda"
] = "pedidos → 0"

In [78]:
analise_produto_top_50["mecanismo_queda"].value_counts()

mecanismo_queda
pedidos → 0                     75
pedidos ↓ + ticket ↑/estável    16
pedidos ↓ + ticket ↓            12
não classificado                 1
Name: count, dtype: int64

In [79]:
analise_produto_top_50[
    analise_produto_top_50["mecanismo_queda"] == "não classificado"
][
    [
        "product_category_name",
        "product_id",
        "2018-05_receita",
        "2018-06_receita",
        "variacao_absoluta",
        "2018-05_pedidos",
        "2018-06_pedidos",
        "variacao_pedidos",
        "ticket_maio",
        "ticket_junho",
        "variacao_ticket"
    ]
]

purchase_month,product_category_name,product_id,2018-05_receita,2018-06_receita,variacao_absoluta,2018-05_pedidos,2018-06_pedidos,variacao_pedidos,ticket_maio,ticket_junho,variacao_ticket
2023,moveis_escritorio,b114bf337c0626166abe574eee9e3f32,1980.22,747.74,-1232.48,4,5,1,495.055,149.548,-345.507


In [80]:
analise_produto_top_50.loc[
    analise_produto_top_50["product_id"] == "b114bf337c0626166abe574eee9e3f32",
    "mecanismo_queda"
] = "pedidos ↑ + ticket ↓"

In [81]:
analise_produto_top_50["mecanismo_queda"].value_counts()

mecanismo_queda
pedidos → 0                     75
pedidos ↓ + ticket ↑/estável    16
pedidos ↓ + ticket ↓            12
pedidos ↑ + ticket ↓             1
Name: count, dtype: int64

In [82]:
sellers_top_produtos = (
    eda_base[
        eda_base["product_id"].isin(
            analise_produto_top_50["product_id"]
        )
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby("seller_id")["product_id"]
    .nunique()
    .sort_values(ascending=False)
)

sellers_top_produtos.head(20)

seller_id
7d13fca15225358621be4086e1eb0964    10
4869f7a5dfa277a7dca6462dcf3b52b2     9
2eb70248d66e0e3ef83659f71b244378     5
966cb4760537b1404caedd472cc610a5     4
fa1c13f2614d7b5c4749cbc52fecda94     4
eeb6de78f79159600292e314a77cbd18     3
de722cd6dad950a92b7d4f82673f8833     3
d921b68bf747894be13a97ae52b0f386     3
955fee9216a65b617aa5c0531780ce60     3
2bf6a2c1e71bbd29a4ad64e6d3c3629f     3
b94cc9f10ddc85e4ba73a6f7974e7101     2
039e6ad9dae79614493083e241147386     2
850f4f8af5ea87287ac68de36e29107f     2
70eea00b476a314817cefde4aad4f89a     2
7c67e1448b00f6e969d365cea6b010ab     2
17a053fcb14bd219540cbde0df490be0     2
1f50f920176fa81dab994f9023523100     2
1025f0e2d44d7041d6cf58b6550e0bfa     2
c6bda72e4dbf5c5866b13cb1810c6d03     2
ba90964cff9b9e0e6f32b23b82465f7b     2
Name: product_id, dtype: int64

In [83]:
perda_seller = (
    eda_base[
        eda_base["product_id"].isin(
            analise_produto_top_50["product_id"]
        )
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby(["seller_id", "purchase_month"])["price"]
    .sum()
    .unstack("purchase_month", fill_value=0)
    .reset_index()
)

perda_seller["variacao_absoluta"] = (
    perda_seller[meses_analise[1]]
    - perda_seller[meses_analise[0]]
)

perda_seller["perda_receita"] = (
    perda_seller["variacao_absoluta"] * -1
)

perda_seller = (
    perda_seller[
        perda_seller["variacao_absoluta"] < 0
    ]
    .sort_values("perda_receita", ascending=False)
)

perda_seller.head(20)

purchase_month,seller_id,2018-05,2018-06,variacao_absoluta,perda_receita
29,7d13fca15225358621be4086e1eb0964,25644.84,2923.99,-22720.85,22720.85
10,2eb70248d66e0e3ef83659f71b244378,12135.30,1433.10,-10702.20,10702.20
33,955fee9216a65b617aa5c0531780ce60,8270.80,179.80,-8091.00,8091.00
34,966cb4760537b1404caedd472cc610a5,8649.95,1568.99,-7080.96,7080.96
44,ba90964cff9b9e0e6f32b23b82465f7b,8832.60,1999.00,-6833.60,6833.60
58,de722cd6dad950a92b7d4f82673f8833,10913.10,4237.20,-6675.90,6675.90
1,039e6ad9dae79614493083e241147386,6480.00,0.00,-6480.00,6480.00
13,4869f7a5dfa277a7dca6462dcf3b52b2,8757.00,2682.50,-6074.50,6074.50
9,2bf6a2c1e71bbd29a4ad64e6d3c3629f,5500.00,0.00,-5500.00,5500.00
50,c72de06d72748d1a0dfb2125be43ba63,5040.00,0.00,-5040.00,5040.00


In [84]:
pedidos_seller = (
    eda_base[
        eda_base["product_id"].isin(
            analise_produto_top_50["product_id"]
        )
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby(["seller_id", "purchase_month"])["order_id"]
    .nunique()
    .unstack("purchase_month", fill_value=0)
    .reset_index()
)

pedidos_seller["variacao_pedidos"] = (
    pedidos_seller[meses_analise[1]]
    - pedidos_seller[meses_analise[0]]
)

pedidos_seller["variacao_pedidos_percentual"] = (
    pedidos_seller["variacao_pedidos"]
    / pedidos_seller[meses_analise[0]]
) * 100

pedidos_seller = (
    pedidos_seller[
        pedidos_seller["variacao_pedidos"] < 0
    ]
    .sort_values("variacao_pedidos")
)

pedidos_seller.head(20)

purchase_month,seller_id,2018-05,2018-06,variacao_pedidos,variacao_pedidos_percentual
29,7d13fca15225358621be4086e1eb0964,181,19,-162,-89.502762
33,955fee9216a65b617aa5c0531780ce60,80,1,-79,-98.750000
10,2eb70248d66e0e3ef83659f71b244378,83,9,-74,-89.156627
8,1f50f920176fa81dab994f9023523100,52,12,-40,-76.923077
13,4869f7a5dfa277a7dca6462dcf3b52b2,57,19,-38,-66.666667
58,de722cd6dad950a92b7d4f82673f8833,59,28,-31,-52.542373
27,7a67c85e85bb2ce8582c35f2203ad736,36,9,-27,-75.000000
3,1025f0e2d44d7041d6cf58b6550e0bfa,20,3,-17,-85.000000
32,8ed8ea9a57c862f2c18c2fe41f601d56,12,0,-12,-100.000000
34,966cb4760537b1404caedd472cc610a5,11,2,-9,-81.818182


In [85]:
seller_produto = (
    eda_base[
        eda_base["product_id"].isin(
            analise_produto_top_50["product_id"]
        )
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby(
        [
            "seller_id",
            "product_id",
            "purchase_month"
        ]
    )
    .agg(
        receita=("price", "sum"),
        pedidos=("order_id", "nunique")
    )
    .reset_index()
)

seller_produto.head(20)

,seller_id,product_id,purchase_month,receita,pedidos
0,0241d4d5d36f10f80c644447315af0bd,698b3ddae2f0b80c2a48fb40624ca4e4,2018-05,1540.0,6
1,0241d4d5d36f10f80c644447315af0bd,698b3ddae2f0b80c2a48fb40624ca4e4,2018-06,749.7,3
2,039e6ad9dae79614493083e241147386,0563d4cc419141eab2e5b438046e844e,2018-05,3930.0,1
3,039e6ad9dae79614493083e241147386,1231bc5d1cab6d3f64357eba69240f63,2018-05,2550.0,1
4,06df0c10d34e459d9672bc77388742eb,0bbdc963004d9b2fd3427ee3c5ae3608,2018-05,2691.0,8
5,06df0c10d34e459d9672bc77388742eb,0bbdc963004d9b2fd3427ee3c5ae3608,2018-06,1296.0,3
6,1025f0e2d44d7041d6cf58b6550e0bfa,5d6bea33648f018dbb563f3a2fab09f3,2018-05,1050.0,7
7,1025f0e2d44d7041d6cf58b6550e0bfa,5d6bea33648f018dbb563f3a2fab09f3,2018-06,150.0,1
8,1025f0e2d44d7041d6cf58b6550e0bfa,962a6951154f98f2c8e9a5b8b2bcf4a9,2018-05,1050.0,13
9,1025f0e2d44d7041d6cf58b6550e0bfa,962a6951154f98f2c8e9a5b8b2bcf4a9,2018-06,150.0,2


In [86]:
seller_produto_pivot = (
    seller_produto
    .pivot_table(
        index=["seller_id", "product_id"],
        columns="purchase_month",
        values=["receita", "pedidos"],
        fill_value=0
    )
)

seller_produto_pivot.columns = [
    f"{metrica}_{mes}"
    for metrica, mes in seller_produto_pivot.columns
]

seller_produto_pivot = seller_produto_pivot.reset_index()

seller_produto_pivot["variacao_receita"] = (
    seller_produto_pivot["receita_2018-06"]
    - seller_produto_pivot["receita_2018-05"]
)

seller_produto_pivot["variacao_pedidos"] = (
    seller_produto_pivot["pedidos_2018-06"]
    - seller_produto_pivot["pedidos_2018-05"]
)

seller_produto_pivot = (
    seller_produto_pivot[
        seller_produto_pivot["variacao_receita"] < 0
    ]
    .sort_values("variacao_receita")
)

seller_produto_pivot.head(20)

,seller_id,product_id,pedidos_2018-05,pedidos_2018-06,receita_2018-05,receita_2018-06,variacao_receita,variacao_pedidos
54,7d13fca15225358621be4086e1eb0964,53b36df67ebb7c41585e8d54d6772e08,100.0,9.0,11001.90,1071.0,-9930.90,-91.0
65,955fee9216a65b617aa5c0531780ce60,aca2eb7d00ea1a7b8ebd4e68314663af,74.0,1.0,6430.80,179.8,-6251.00,-73.0
91,c72de06d72748d1a0dfb2125be43ba63,af751fe756699bddfce4fdd67242fc2a,3.0,0.0,5040.00,0.0,-5040.00,-3.0
9,17e34d8224d27a541263c4c64b11a56b,0abc19e41fdc9d6894b8cae651619253,2.0,0.0,4941.00,0.0,-4941.00,-2.0
56,7d13fca15225358621be4086e1eb0964,6f3b5b605d91b7439c5e3f5a8dffeea7,30.0,0.0,4355.00,0.0,-4355.00,-30.0
67,966cb4760537b1404caedd472cc610a5,57c9362442482cd2451f2eabf468e532,5.0,1.0,4955.97,869.0,-4086.97,-4.0
16,2eb70248d66e0e3ef83659f71b244378,89b121bee266dcd25688a1ba72eefb61,46.0,0.0,3995.00,0.0,-3995.00,-46.0
1,039e6ad9dae79614493083e241147386,0563d4cc419141eab2e5b438046e844e,1.0,0.0,3930.00,0.0,-3930.00,-1.0
102,de722cd6dad950a92b7d4f82673f8833,3dd2a17168ec895c781a9191c1e95ad7,44.0,27.0,7794.80,4047.3,-3747.50,-17.0
83,ba90964cff9b9e0e6f32b23b82465f7b,ff92ca9bb0b3f4ec00a9b76c9f68cb3a,4.0,0.0,3435.60,0.0,-3435.60,-4.0


In [87]:
perdas_seller_produto = seller_produto_pivot.copy()

perdas_seller_produto["perda_receita"] = (
    perdas_seller_produto["variacao_receita"] * -1
)

perdas_seller_produto = (
    perdas_seller_produto[
        perdas_seller_produto["perda_receita"] > 0
    ]
    .sort_values("perda_receita", ascending=False)
)

perdas_seller_produto["participacao_perdas"] = (
    perdas_seller_produto["perda_receita"]
    / perdas_seller_produto["perda_receita"].sum()
) * 100

perdas_seller_produto["participacao_acumulada"] = (
    perdas_seller_produto["participacao_perdas"]
    .cumsum()
)

perdas_seller_produto.head(20)

,seller_id,product_id,pedidos_2018-05,pedidos_2018-06,receita_2018-05,receita_2018-06,variacao_receita,variacao_pedidos,perda_receita,participacao_perdas,participacao_acumulada
54,7d13fca15225358621be4086e1eb0964,53b36df67ebb7c41585e8d54d6772e08,100.0,9.0,11001.90,1071.0,-9930.90,-91.0,9930.90,5.873963,5.873963
65,955fee9216a65b617aa5c0531780ce60,aca2eb7d00ea1a7b8ebd4e68314663af,74.0,1.0,6430.80,179.8,-6251.00,-73.0,6251.00,3.697363,9.571325
91,c72de06d72748d1a0dfb2125be43ba63,af751fe756699bddfce4fdd67242fc2a,3.0,0.0,5040.00,0.0,-5040.00,-3.0,5040.00,2.981076,12.552402
9,17e34d8224d27a541263c4c64b11a56b,0abc19e41fdc9d6894b8cae651619253,2.0,0.0,4941.00,0.0,-4941.00,-2.0,4941.00,2.922520,15.474921
56,7d13fca15225358621be4086e1eb0964,6f3b5b605d91b7439c5e3f5a8dffeea7,30.0,0.0,4355.00,0.0,-4355.00,-30.0,4355.00,2.575910,18.050831
67,966cb4760537b1404caedd472cc610a5,57c9362442482cd2451f2eabf468e532,5.0,1.0,4955.97,869.0,-4086.97,-4.0,4086.97,2.417375,20.468206
16,2eb70248d66e0e3ef83659f71b244378,89b121bee266dcd25688a1ba72eefb61,46.0,0.0,3995.00,0.0,-3995.00,-46.0,3995.00,2.362976,22.831183
1,039e6ad9dae79614493083e241147386,0563d4cc419141eab2e5b438046e844e,1.0,0.0,3930.00,0.0,-3930.00,-1.0,3930.00,2.324530,25.155712
102,de722cd6dad950a92b7d4f82673f8833,3dd2a17168ec895c781a9191c1e95ad7,44.0,27.0,7794.80,4047.3,-3747.50,-17.0,3747.50,2.216584,27.372297
83,ba90964cff9b9e0e6f32b23b82465f7b,ff92ca9bb0b3f4ec00a9b76c9f68cb3a,4.0,0.0,3435.60,0.0,-3435.60,-4.0,3435.60,2.032100,29.404397


In [88]:
impacto_mecanismos = (
    analise_produto_top_50
    .groupby("mecanismo_queda")
    .agg(
        produtos=("product_id", "count"),
        perda_receita=("variacao_absoluta", lambda x: -x.sum()),
        perda_pedidos=("variacao_pedidos", lambda x: -x.sum())
    )
    .sort_values("perda_receita", ascending=False)
)

impacto_mecanismos

,produtos,perda_receita,perda_pedidos
mecanismo_queda,,,
pedidos → 0,75,103613.05,221
pedidos ↓ + ticket ↑/estável,16,40389.07,318
pedidos ↓ + ticket ↓,12,23042.05,93
pedidos ↑ + ticket ↓,1,1232.48,-1


In [89]:
pedidos_operacao = (
    eda_base[
        eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby("purchase_month")["order_id"]
    .nunique()
)

pedidos_produtos_impacto = (
    eda_base[
        eda_base["product_id"].isin(
            analise_produto_top_50["product_id"]
        )
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby("purchase_month")["order_id"]
    .nunique()
)

comparacao_pedidos = pd.DataFrame({
    "pedidos_operacao": pedidos_operacao,
    "pedidos_produtos_impacto": pedidos_produtos_impacto
})

comparacao_pedidos["variacao_operacao"] = (
    comparacao_pedidos["pedidos_operacao"].diff()
)

comparacao_pedidos["variacao_produtos_impacto"] = (
    comparacao_pedidos["pedidos_produtos_impacto"].diff()
)

comparacao_pedidos

,pedidos_operacao,pedidos_produtos_impacto,variacao_operacao,variacao_produtos_impacto
purchase_month,,,,
2018-05,6853,771,NaN,NaN
2018-06,6160,144,-693.0,-627.0


In [90]:
receita_produtos_impacto = (
    eda_base[
        eda_base["product_id"].isin(
            analise_produto_top_50["product_id"]
        )
        & eda_base["purchase_month"].isin(meses_analise)
    ]
    .groupby("purchase_month")["price"]
    .sum()
)

comparacao_receita = pd.DataFrame({
    "receita_operacao": receita_mensal.loc[meses_analise],
    "receita_produtos_impacto": receita_produtos_impacto
})

comparacao_receita["variacao_operacao"] = (
    comparacao_receita["receita_operacao"].diff()
)

comparacao_receita["variacao_produtos_impacto"] = (
    comparacao_receita["receita_produtos_impacto"].diff()
)

comparacao_receita

,receita_operacao,receita_produtos_impacto,variacao_operacao,variacao_produtos_impacto
2018-05,996517.68,193890.27,NaN,NaN
2018-06,865124.31,25613.62,-131393.37,-168276.65


In [91]:
resumo_eda = pd.DataFrame({
    "indicador": [
        "Queda de receita da operação",
        "Queda de pedidos da operação",
        "Produtos de maior impacto",
        "Perda de receita dos produtos de maior impacto",
        "Queda de pedidos dos produtos de maior impacto",
        "Participação dos produtos de impacto na queda de pedidos",
        "Produtos com pedidos zerados em junho"
    ],
    "valor": [
        f"{comparacao_receita.loc[meses_analise[1], 'variacao_operacao']:,.2f}",
        f"{comparacao_pedidos.loc[meses_analise[1], 'variacao_operacao']:,.0f}",
        len(analise_produto_top_50),
        f"{comparacao_receita.loc[meses_analise[1], 'variacao_produtos_impacto']:,.2f}",
        f"{comparacao_pedidos.loc[meses_analise[1], 'variacao_produtos_impacto']:,.0f}",
        f"{(
            comparacao_pedidos.loc[
                meses_analise[1],
                "variacao_produtos_impacto"
            ]
            / comparacao_pedidos.loc[
                meses_analise[1],
                "variacao_operacao"
            ]
        ) * 100:.2f}%",
        (
            analise_produto_top_50["mecanismo_queda"]
            .eq("pedidos → 0")
            .sum()
        )
    ]
})

resumo_eda

,indicador,valor
0,Queda de receita da operação,"-131,393.37"
1,Queda de pedidos da operação,-693
2,Produtos de maior impacto,104
3,Perda de receita dos produtos de maior impacto,"-168,276.65"
4,Queda de pedidos dos produtos de maior impacto,-627
5,Participação dos produtos de impacto na queda ...,90.48%
6,Produtos com pedidos zerados em junho,75


# Consolidação da Análise Exploratória de Dados (EDA)

## 1. Objetivo da EDA

A Análise Exploratória de Dados teve como objetivo investigar onde e como a deterioração observada entre maio e junho de 2018 se manifesta no desempenho comercial da operação.

A análise partiu do fenômeno identificado durante o Data Profiling:

- redução de 13,19% na receita;
- redução de 10,11% no número de pedidos;
- redução de 3,42% no ticket médio.

A investigação foi conduzida de forma progressiva, partindo do nível agregado da operação e avançando para categorias, produtos e relações entre produtos e sellers.

Os resultados desta etapa são exploratórios e descritivos. Eles identificam padrões e associações presentes nos dados, mas não estabelecem relações de causalidade.

---

## 2. Reconfirmação do fenômeno

A comparação entre maio e junho de 2018 confirmou a deterioração recente identificada no Data Profiling.

A receita de produtos passou de R$ 996.517,68 para R$ 865.124,31, representando uma redução líquida de R$ 131.393,37 (-13,19%).

No mesmo período:

- os pedidos passaram de 6.853 para 6.160, uma redução de 693 pedidos (-10,11%);
- o ticket médio passou de R$ 145,41 para R$ 140,44, uma redução de 3,42%.

A decomposição da receita foi preservada pela relação:

**Receita = Pedidos × Ticket médio**

Portanto, a deterioração agregada ocorreu simultaneamente nos dois componentes da receita, com maior variação no volume de pedidos.

---

## 3. Localização da deterioração por categoria

A análise por categoria mostrou que a redução da receita não ocorreu de maneira uniforme.

Foram priorizadas as dez categorias que apresentaram as maiores perdas absolutas de receita entre maio e junho de 2018.

As principais perdas absolutas foram observadas em:

- `relogios_presentes`;
- `ferramentas_jardim`;
- `esporte_lazer`;
- `moveis_decoracao`;
- `cool_stuff`.

As dez categorias prioritárias apresentaram, em conjunto, perda bruta de R$ 135.440,90.

Esse valor é superior à redução líquida da operação, de R$ 131.393,37, porque outras categorias apresentaram crescimento no mesmo período e compensaram parcialmente as perdas observadas nas categorias selecionadas.

A análise também mostrou que os mecanismos de deterioração diferem entre categorias. Em algumas delas, a redução da receita ocorreu acompanhada por queda no número de pedidos e no ticket médio; em outras, houve redução de pedidos acompanhada por aumento do ticket.

Esse comportamento reforça a necessidade de analisar conjuntamente volume de pedidos e ticket médio.

---

## 4. Concentração da deterioração por produto

A investigação foi aprofundada nos produtos pertencentes às dez categorias prioritárias.

Foram identificados 2.413 produtos com participação nas vendas dessas categorias durante o período analisado.

Entre os produtos que apresentaram redução de receita, 104 produtos foram suficientes para atingir aproximadamente 50% da perda acumulada observada nesse conjunto.

Esses 104 produtos apresentaram:

- perda líquida de receita de R$ 168.276,65;
- redução de 627 pedidos.

A perda observada nesses produtos foi superior à perda líquida da operação, indicando que outros produtos compensaram parcialmente essa deterioração por meio de crescimento de receita no mesmo período.

---

## 5. Mecanismos observados nos produtos de maior impacto

Os 104 produtos de maior impacto foram classificados de acordo com o comportamento de pedidos e ticket médio entre maio e junho de 2018.

| Mecanismo observado | Produtos | Perda de receita |
|---|---:|---:|
| Pedidos → 0 | 75 | R$ 103.613,05 |
| Pedidos ↓ + ticket ↑/estável | 16 | R$ 40.389,07 |
| Pedidos ↓ + ticket ↓ | 12 | R$ 23.042,05 |
| Pedidos ↑ + ticket ↓ | 1 | R$ 1.232,48 |

O principal padrão observado foi a redução do volume de pedidos.

Dos 104 produtos:

- 75 deixaram de registrar pedidos em junho;
- 16 apresentaram redução de pedidos acompanhada por aumento ou estabilidade do ticket;
- 12 apresentaram redução simultânea de pedidos e ticket;
- 1 apresentou aumento de pedidos acompanhado por redução do ticket.

Considerando os produtos classificados nos mecanismos acima, a redução do volume de pedidos aparece como o principal padrão associado à perda de receita observada.

Esse resultado deve ser interpretado como evidência exploratória e não como demonstração de causalidade.

---

## 6. Participação dos produtos de maior impacto na queda de pedidos

A operação apresentou redução de 693 pedidos entre maio e junho de 2018.

Os 104 produtos de maior impacto apresentaram redução de 627 pedidos no mesmo período.

Assim, esses produtos concentraram aproximadamente 90,48% da redução líquida de pedidos observada na operação.

Esse resultado reforça a relevância desses produtos para a investigação da deterioração recente.

Entretanto, a participação na variação líquida não deve ser interpretada como participação causal. O resultado indica concentração da redução observada, e não que esses produtos sejam necessariamente a causa da queda.

---

## 7. Investigação de sellers

A análise seller × produto mostrou que parte relevante das perdas está distribuída em combinações específicas de sellers e produtos.

Foram identificadas combinações seller × produto com perdas expressivas de receita e, em diversos casos, redução simultânea no número de pedidos.

Os resultados indicam que a dimensão seller pode contribuir para explicar parte da heterogeneidade observada entre os produtos.

Entretanto, os resultados exploratórios não são suficientes para afirmar que determinados sellers causaram a deterioração.

Por esse motivo, a dimensão seller será considerada como uma possível variável explicativa nas etapas posteriores, caso os testes estatísticos e análises adicionais forneçam evidências suficientes.

---

## 8. Principais achados da EDA

A investigação exploratória permitiu estabelecer os seguintes achados:

1. A operação apresentou uma deterioração recente entre maio e junho de 2018, com redução simultânea de receita, pedidos e ticket médio.

2. A deterioração não ocorreu de maneira uniforme entre as categorias.

3. Dez categorias concentraram as maiores perdas absolutas de receita no período analisado.

4. Dentro dessas categorias, 104 produtos concentraram aproximadamente 50% das perdas observadas entre os produtos com queda.

5. Esses 104 produtos apresentaram perda líquida de R$ 168.276,65 e redução de 627 pedidos.

6. A redução de pedidos representou o principal mecanismo observado entre os produtos de maior impacto.

7. Os 104 produtos concentraram aproximadamente 90,48% da redução líquida de pedidos observada na operação.

8. A perda dos produtos de maior impacto foi parcialmente compensada pelo crescimento de outros produtos da operação.

9. A dimensão seller apresentou heterogeneidade relevante nas perdas observadas, justificando sua consideração nas análises posteriores.

10. Os padrões identificados são exploratórios e não permitem estabelecer causalidade.

---

## 9. Hipóteses para a análise estatística

Com base nos padrões identificados durante a EDA, foram formuladas hipóteses que poderão ser investigadas estatisticamente.

### Hipótese 1 — alteração no volume de pedidos

A distribuição do número de pedidos dos produtos de maior impacto pode ter apresentado alteração estatisticamente significativa entre maio e junho de 2018.

**Variável principal:** número de pedidos por produto.

---

### Hipótese 2 — alteração no ticket médio

A distribuição do ticket médio dos produtos de maior impacto pode ter apresentado alteração estatisticamente significativa entre maio e junho de 2018.

**Variável principal:** ticket médio por produto.

Essa hipótese deve ser analisada considerando que produtos sem pedidos em junho não possuem ticket médio definido nesse período.

---

### Hipótese 3 — heterogeneidade entre categorias

A magnitude da deterioração pode diferir entre as categorias prioritárias, indicando que determinadas categorias apresentam comportamento distinto das demais.

**Variáveis principais:**

- variação de receita;
- variação de pedidos;
- variação do ticket médio.

---

### Hipótese 4 — heterogeneidade entre sellers

A magnitude da deterioração pode variar entre sellers que comercializam os produtos de maior impacto.

**Variáveis principais:**

- variação de receita;
- variação de pedidos;
- seller;
- produto.

Essa hipótese será avaliada somente se a estrutura e o volume dos dados permitirem uma comparação estatisticamente adequada.

---

## 10. Limitações da análise exploratória

Os resultados devem ser interpretados considerando as seguintes limitações:

- a comparação principal utiliza maio e junho de 2018, ambos meses completos;
- setembro de 2016 e setembro de 2018 possuem cobertura parcial e não foram utilizados na comparação principal;
- categorias podem aparecer em um mesmo pedido, portanto seus números de pedidos não devem ser somados para representar pedidos únicos da operação;
- produtos também podem aparecer em diferentes pedidos e sellers;
- a análise de seller × produto é necessária porque um mesmo produto pode ser comercializado por diferentes sellers;
- grandes variações percentuais podem ocorrer em produtos com baixo volume de vendas;
- produtos que passaram a apresentar zero pedidos em junho não possuem ticket médio definido nesse mês;
- os 104 produtos representam o conjunto de maior impacto dentro das dez categorias prioritárias, e não todos os produtos da operação;
- os resultados da EDA representam padrões observados nos dados e não evidências de causalidade.

---

## 11. Encerramento da EDA

A EDA permitiu transformar a deterioração agregada observada no Data Profiling em um conjunto de padrões mais específicos.

A evidência exploratória aponta para uma forte concentração da redução de pedidos nos produtos de maior impacto, acompanhada por perdas relevantes de receita.

A partir desses resultados, a investigação deixa de buscar apenas onde ocorreu a queda e passa a avaliar se os padrões observados são estatisticamente consistentes.

A próxima etapa será a **Análise Estatística**, na qual as hipóteses formuladas nesta etapa serão avaliadas por meio de métodos estatísticos adequados à estrutura dos dados.

O objetivo será distinguir padrões observados de evidências estatísticas, mantendo a separação entre associação e causalidade.